<a href="https://colab.research.google.com/github/Ezechis/pidgin-sentiment/blob/main/notebooks/train_pidgin_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pidgin Sentiment Engine: AfriBERTa fine-tuning on NaijaSenti (Nigerian Pidgin)

**What this notebook does** (maps to the Technical Mini-Thesis framework):

| Step | Thesis section |
|---|---|
| 1. Load the official NaijaSenti `pcm` splits | 3.1 Data Source |
| 2. Clean, de-duplicate, check train/test leakage | 3.1 Preprocessing |
| 3. Exploratory analysis (class balance, lengths) | 3.1 / 4.1 |
| 4. Classical baseline: TF-IDF + Logistic Regression | 2.2 / 4.1 |
| 5. Fine-tune mBERT and AfriBERTa-large (class-weighted) | 3.2 |
| 6. Test-set metrics, confusion matrices, learning curves | 4.1 |
| 7. Error analysis: where the model breaks | 4.3 |
| 8. Save the best model to Drive and publish it to the Hugging Face Hub | 3.4 |
| 9. Deploy the web app as a Hugging Face Space | 3.4 / Appendix |

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**.

To publish the model and web app (steps 8-9), add a Hugging Face **write** token in Colab's 🔑 *Secrets* panel
under the name `HF_TOKEN`, and enable notebook access. Without it, everything else still runs.

In [1]:
# ---- 0. Setup -------------------------------------------------------------
import os, sys, json, random, csv, time
IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("PIDGIN_SMOKE") == "1"   # local CPU smoke test only

if IN_COLAB:
    !pip install -q -U transformers accelerate sentencepiece huggingface_hub

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

LABELS = ["negative", "neutral", "positive"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

MODELS = {
    "mBERT": "bert-base-multilingual-cased",
    "AfriBERTa-large": "castorini/afriberta_large",
}
HPARAMS = dict(epochs=4, lr=2e-5, batch_size=32, max_len=128, weight_decay=0.01, warmup_ratio=0.1)

# Hugging Face repo that the web app loads the model from.
HF_REPO_ID = "ezechinnabugwu/pidgin-sentiment-model"

if SMOKE:
    MODELS = {"tiny-bert": "hf-internal-testing/tiny-random-BertForSequenceClassification"}
    HPARAMS.update(epochs=1, batch_size=8, max_len=32)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/Pidgin_Slang_NLP/v2_real_data"
else:
    OUT_DIR = os.path.abspath("outputs_smoke" if SMOKE else "outputs")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | torch {torch.__version__} | outputs -> {OUT_DIR}")
if IN_COLAB and DEVICE != "cuda":
    print("⚠️  No GPU. Switch the runtime to T4 GPU, or training will take hours.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 39.6 MB/s eta 0:00:00


ValueError: mount failed

## 1. Data source: NaijaSenti (Muhammad et al., LREC 2022)
The official Nigerian Pidgin (`pcm`) split, human-annotated tweets, pulled directly from the
authors' GitHub repository. (The Hugging Face mirror `HausaNLP/NaijaSenti-Twitter` relies on a
loading script that current `datasets` versions no longer execute, so we read the source TSVs.)

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/hausanlp/NaijaSenti/main/data/annotated_tweets/pcm/{}.tsv"
LOCAL_DATA = os.environ.get("PIDGIN_DATA_DIR")  # optional offline copy

def load_split(name):
    src = os.path.join(LOCAL_DATA, f"pcm_{name}.tsv") if LOCAL_DATA else BASE_URL.format(name)
    df = pd.read_csv(src, sep="\t", quoting=csv.QUOTE_NONE, dtype=str)
    return df.rename(columns={"tweet": "text"})[["text", "label"]]

raw = {s: load_split(s) for s in ["train", "dev", "test"]}
for s, df in raw.items():
    print(f"{s:5s} {len(df):5d} rows  {df['label'].value_counts().to_dict()}")
raw["train"].sample(5, random_state=SEED)

## 2. Preprocessing
The tweets arrive already lower-cased with user handles, URLs and most punctuation removed by
the dataset authors. We deliberately **do not** remove stop-words or stem: words like *dey*,
*no*, *go*, *don* carry Pidgin grammar (tense, negation, aspect). We only:
1. normalise whitespace and drop empty rows,
2. remove exact duplicates inside each split,
3. remove any test/dev tweet that also appears in train (prevents leakage and inflated scores).

In [ ]:
def clean(df):
    df = df.copy()
    df["text"] = df["text"].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()
    df = df[(df["text"] != "") & df["label"].isin(LABELS)]
    return df.drop_duplicates(subset="text").reset_index(drop=True)

conflicts = raw["train"].groupby("text")["label"].nunique()
print(f"train tweets duplicated with conflicting labels: {(conflicts > 1).sum()}")

data = {s: clean(df) for s, df in raw.items()}
official_test = data["test"].copy()   # kept only to compare with published NaijaSenti scores
train_texts = set(data["train"]["text"])
for s in ["dev", "test"]:
    before = len(data[s])
    data[s] = data[s][~data[s]["text"].isin(train_texts)].reset_index(drop=True)
    print(f"{s}: removed {before - len(data[s])} tweets that also appear in train")

prep_log = pd.DataFrame({s: [len(raw[s]), len(data[s])] for s in data}, index=["raw", "clean"])
print(prep_log)
if SMOKE:
    data = {s: df.groupby("label", group_keys=False).head(20).reset_index(drop=True)
            for s, df in data.items()}
    official_test = official_test.groupby("label", group_keys=False).head(20).reset_index(drop=True)
for df in [*data.values(), official_test]:
    df["y"] = df["label"].map(LABEL2ID)

## 3. Exploratory analysis
Key finding: **neutral** is only ~1.4% of training data but ~13% of the cleaned test set. A model that
ignores neutral can still reach high accuracy, which is why we report **macro-F1** as the headline
metric and train with class weights.

In [ ]:
dist = pd.DataFrame({s: df["label"].value_counts(normalize=True).reindex(LABELS).fillna(0) * 100
                     for s, df in data.items()})
lengths = data["train"]["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
dist.T.plot(kind="bar", stacked=True, ax=axes[0], color=["#d9534f", "#f0ad4e", "#5cb85c"])
axes[0].set_title("Class distribution per split (%)"); axes[0].set_ylabel("%"); axes[0].tick_params(axis="x", rotation=0)
axes[1].hist(lengths, bins=40, color="#4a6fa5")
axes[1].set_title(f"Tweet length in words (train), median={int(lengths.median())}"); axes[1].set_xlabel("words")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "eda_distribution.png"), dpi=150); plt.show()
print(dist.round(1))

## 4. Baseline: TF-IDF + Logistic Regression
Word n-grams plus **character n-grams (2–5)**. Character n-grams partly absorb Pidgin's
non-standard spelling (*shege / shegeh / sege*). This baseline tells us how much the
pre-trained transformers actually add.

In [ ]:
def score(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred), "precision_macro": p, "recall_macro": r,
            "f1_macro": f, "f1_weighted": precision_recall_fscore_support(
                y_true, y_pred, average="weighted", zero_division=0)[2]}

t0 = time.time()
baseline = make_pipeline(
    make_union(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
               TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    LogisticRegression(max_iter=2000, class_weight="balanced", C=2.0, random_state=SEED),
)
baseline.fit(data["train"]["text"], data["train"]["y"])
predictions = {"TF-IDF + LogReg": baseline.predict(data["test"]["text"])}
train_minutes = {"TF-IDF + LogReg": (time.time() - t0) / 60}
results = {"TF-IDF + LogReg": score(data["test"]["y"], predictions["TF-IDF + LogReg"])}
official_f1 = {"TF-IDF + LogReg": score(official_test["y"], baseline.predict(official_test["text"]))["f1_macro"]}
print(json.dumps(results, indent=2))

## 5. Fine-tuning pre-trained encoders
Both models get an identical recipe so the comparison is fair: a new 3-way classification head,
AdamW, linear warm-up, 4 epochs, and **class-weighted cross-entropy** (inverse square-root
frequency, which boosts neutral without destabilising training). The checkpoint with the best
**dev macro-F1** is kept. Train metrics are computed on a fixed 1,000-tweet sample each epoch so we
can plot training vs validation curves.

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, DataCollatorWithPadding)

counts = data["train"]["y"].value_counts().reindex(range(len(LABELS))).fillna(1).values
class_weights = 1.0 / np.sqrt(counts)
class_weights = torch.tensor(class_weights / class_weights.mean(), dtype=torch.float)
print("Class weights:", dict(zip(LABELS, class_weights.numpy().round(2))))

class TorchDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.enc = tokenizer(list(df["text"]), truncation=True, max_length=max_len)
        self.y = list(df["y"])
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.y[i]
        return item

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def hf_metrics(eval_pred):
    logits, labels = eval_pred
    return score(labels, np.argmax(logits, axis=-1))

train_probe = data["train"].sample(min(1000, len(data["train"])), random_state=SEED)
histories, trained = {}, {}

def fine_tune(name, checkpoint):
    tok = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
        ignore_mismatched_sizes=True)
    ds = {s: TorchDataset(df, tok, HPARAMS["max_len"]) for s, df in data.items()}
    total_steps = int(np.ceil(len(ds["train"]) / HPARAMS["batch_size"])) * HPARAMS["epochs"]
    warmup_steps = int(total_steps * HPARAMS["warmup_ratio"])
    args = TrainingArguments(
        output_dir=os.path.join("/tmp" if IN_COLAB else OUT_DIR, "ckpt", name),
        num_train_epochs=HPARAMS["epochs"], learning_rate=HPARAMS["lr"],
        per_device_train_batch_size=HPARAMS["batch_size"], per_device_eval_batch_size=64,
        weight_decay=HPARAMS["weight_decay"], warmup_steps=warmup_steps,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="eval_dev_f1_macro", greater_is_better=True,
        logging_strategy="steps", logging_steps=20, report_to="none", seed=SEED,
        fp16=(DEVICE == "cuda"), use_cpu=(DEVICE == "cpu"),
    )
    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=ds["train"],
        eval_dataset={"train": TorchDataset(train_probe, tok, HPARAMS["max_len"]), "dev": ds["dev"]},
        processing_class=tok, data_collator=DataCollatorWithPadding(tok), compute_metrics=hf_metrics,
    )
    t0 = time.time()
    trainer.train()
    train_minutes[name] = (time.time() - t0) / 60
    logits = trainer.predict(ds["test"]).predictions
    predictions[name] = np.argmax(logits, axis=-1)
    results[name] = score(data["test"]["y"], predictions[name])
    official_logits = trainer.predict(TorchDataset(official_test, tok, HPARAMS["max_len"])).predictions
    official_f1[name] = score(official_test["y"], np.argmax(official_logits, axis=-1))["f1_macro"]
    histories[name] = pd.DataFrame(trainer.state.log_history)
    trained[name] = (trainer, tok)
    print(f"\n{name}: {train_minutes[name]:.1f} min | test macro-F1 = {results[name]['f1_macro']:.4f}")

for name, ckpt in MODELS.items():
    fine_tune(name, ckpt)

## 6. Results on the held-out test set
Headline metric: **macro-F1** (every class counts equally, so the rare neutral class can't be ignored).

All headline numbers use the **cleaned** test set (no tweets shared with train). The column
`f1_macro_official_split` scores the untouched official test split only so results can be compared
with published NaijaSenti numbers. It is inflated by the train/test overlap removed in step 2.

In [ ]:
params = {"TF-IDF + LogReg": "n/a"}
for name, (trainer, _) in trained.items():
    params[name] = f"{sum(p.numel() for p in trainer.model.parameters()) / 1e6:.0f}M"
summary = pd.DataFrame(results).T
summary["f1_macro_official_split"] = pd.Series(official_f1)
summary["parameters"] = pd.Series(params)
summary["train_minutes"] = pd.Series(train_minutes).round(1)
summary = summary.sort_values("f1_macro", ascending=False)
summary.to_csv(os.path.join(OUT_DIR, "test_results.csv"))
BEST = max(trained, key=lambda n: results[n]["f1_macro"])
print(f"Best fine-tuned model: {BEST}\n")
summary

In [ ]:
fig, axes = plt.subplots(1, len(predictions), figsize=(5.5 * len(predictions), 4.5))
axes = np.atleast_1d(axes)
for ax, (name, pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(data["test"]["y"], pred, labels=range(len(LABELS)), normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
    ax.set_title(f"{name}\nmacro-F1 = {results[name]['f1_macro']:.3f}")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "confusion_matrices.png"), dpi=150); plt.show()

for name, pred in predictions.items():
    print(f"=== {name} ===")
    print(classification_report(data["test"]["y"], pred, labels=range(len(LABELS)),
                                target_names=LABELS, digits=3, zero_division=0))

In [ ]:
# Learning curves: training vs validation loss and macro-F1 per epoch
fig, axes = plt.subplots(len(histories), 2, figsize=(12, 4 * len(histories)), squeeze=False)
for row, (name, h) in enumerate(histories.items()):
    ev = h.dropna(subset=["eval_dev_loss"]).groupby("epoch").last()
    tr = h.dropna(subset=["eval_train_loss"]).groupby("epoch").last()
    axes[row, 0].plot(tr.index, tr["eval_train_loss"], "o-", label="train")
    axes[row, 0].plot(ev.index, ev["eval_dev_loss"], "o-", label="validation")
    axes[row, 0].set_title(f"{name}: loss"); axes[row, 0].set_xlabel("epoch"); axes[row, 0].legend()
    axes[row, 1].plot(tr.index, tr["eval_train_f1_macro"], "o-", label="train macro-F1")
    axes[row, 1].plot(ev.index, ev["eval_dev_f1_macro"], "o-", label="validation macro-F1")
    axes[row, 1].plot(ev.index, ev["eval_dev_accuracy"], "s--", label="validation accuracy")
    axes[row, 1].set_title(f"{name}: accuracy / F1"); axes[row, 1].set_xlabel("epoch"); axes[row, 1].legend()
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "learning_curves.png"), dpi=150); plt.show()

## 7. Error analysis (thesis §4.3: failure modes)
Every misclassified test tweet from the best model is saved to `errors_<model>.csv`.
Read through them. Typical failure modes to look for and write up: sarcasm, mixed sentiment in one
tweet, neutral news-style tweets, religious/proverbial phrasing, and heavy code-switching.

In [ ]:
trainer, tok = trained[BEST]
err = data["test"][["text", "label"]].copy()
err["predicted"] = [ID2LABEL[int(i)] for i in predictions[BEST]]
err = err[err["label"] != err["predicted"]]
err.to_csv(os.path.join(OUT_DIR, f"errors_{BEST}.csv"), index=False)
print(f"{len(err)} misclassified of {len(data['test'])} test tweets")
print(err.groupby(["label", "predicted"]).size().sort_values(ascending=False).rename("count").to_string())
pd.set_option("display.max_colwidth", 160)
err.sample(min(15, len(err)), random_state=SEED)

In [ ]:
# Probe with hand-written slang sentences (illustration only, NOT part of the evaluation)
from transformers import pipeline
clf = pipeline("text-classification", model=trainer.model, tokenizer=tok, top_k=None,
               device=0 if DEVICE == "cuda" else -1)
probes = [
    "sapa dey choke me since morning this economy heavy",
    "abeg track my order delivery rider dey use me play",
    "opay features dey sweet soft work always",
    "this app don cast completely i dey delete am",
    "e choke omo this new album na fire",
    "i just dey check my account balance",
]
for text, scores in zip(probes, clf(probes)):
    top = max(scores, key=lambda s: s["score"])
    print(f"{top['label']:>8s} {top['score']:.2f} | {text}")

## 8. Save and publish the best model
Saved to Drive (backup) and pushed to the Hugging Face Hub so the web app can load it.

In [ ]:
best_dir = os.path.join(OUT_DIR, f"best_model_{BEST}")
trainer.save_model(best_dir); tok.save_pretrained(best_dir)
report = {"best_model": BEST, "checkpoint": MODELS.get(BEST), "hyperparameters": HPARAMS,
          "class_weights": dict(zip(LABELS, class_weights.numpy().round(4).tolist())),
          "rows": {s: len(df) for s, df in data.items()}, "test_results": summary.to_dict(orient="index")}
with open(os.path.join(OUT_DIR, "metrics.json"), "w") as f:
    json.dump(report, f, indent=2, default=str)
print("Saved to", best_dir)

token = None
if IN_COLAB:
    from google.colab import userdata
    try:
        token = userdata.get("HF_TOKEN")
    except Exception as e:
        print(f"HF_TOKEN secret not available ({type(e).__name__}); skipping Hub upload.")
if token and "REPLACE_WITH" not in HF_REPO_ID and not SMOKE:
    trainer.model.push_to_hub(HF_REPO_ID, token=token, commit_message=f"{BEST} fine-tuned on NaijaSenti pcm")
    tok.push_to_hub(HF_REPO_ID, token=token)
    print(f"✅ Published: https://huggingface.co/{HF_REPO_ID}")
else:
    print("Hub upload skipped (set HF_REPO_ID and the HF_TOKEN secret to publish).")

## 9. Deploy the web app (Hugging Face Space)
Creates/updates a free public Gradio Space that loads the model published above.
The app files are pulled from the project's GitHub repository.

In [ ]:
SPACE_ID = "ezechinnabugwu/pidgin-sentiment"
GITHUB_RAW = "https://raw.githubusercontent.com/Ezechis/pidgin-sentiment/main/app/{}"

if token and not SMOKE:
    import urllib.request
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    api.create_repo(SPACE_ID, repo_type="space", space_sdk="gradio", exist_ok=True)
    space_dir = "/tmp/space"
    os.makedirs(space_dir, exist_ok=True)
    for fname in ["app.py", "rules.py", "preprocess.py", "requirements.txt", "README.md"]:
        urllib.request.urlretrieve(GITHUB_RAW.format(fname), os.path.join(space_dir, fname))
    api.upload_folder(folder_path=space_dir, repo_id=SPACE_ID, repo_type="space",
                      commit_message="Deploy Pidgin Sentiment Engine")
    print(f"✅ Web app deploying at: https://huggingface.co/spaces/{SPACE_ID}  (first build takes ~3-5 min)")
else:
    print("Space deploy skipped (needs the HF_TOKEN secret).")